In [2]:
# Importing Necessary Libraries
import numpy as np      
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
# Loding the dataset
df = pd.read_csv("D:\HOPE AI\ML model Failure Prediction System\Datasets\Raw\Predictive_Maintainance_Raw.csv")
print("Original shape:", df.shape)
df.head()

Original shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
# 1. Temperature difference (Machine Temp - Air Temp)
df['Temp_Diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

# 2. Power (RPM * Torque)
df['Power'] = df['Rotational speed [rpm]'] * df['Torque [Nm]']

# 3. Tool Overstrain (Torque * Tool wear)
df['Tool_Strain'] = df['Torque [Nm]'] * df['Tool wear [min]']

df[['Temp_Diff', 'Power', 'Tool_Strain']].head()

,Temp_Diff,Power,Tool_Strain
0,10.5,66382.8,0.0
1,10.5,65190.4,138.9
2,10.4,74001.2,247.0
3,10.4,56603.5,276.5
4,10.5,56320.0,360.0


In [ ]:
# 1. Convert categorical to numerical using one-hot encoding
df = pd.get_dummies(df, columns=['Type'], drop_first=True, dtype=int)

# 2. Drop ID columns and individual failure columns
columns_to_drop = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df = df.drop(columns=columns_to_drop)

print("Remaining columns:")
print(df.columns.tolist())

Remaining columns:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'Temp_Diff', 'Power', 'Tool_Strain', 'Type_L', 'Type_M']


In [ ]:
# Separate features and target
X = df.drop(columns=['Machine failure'])
y = df['Machine failure']

# Step 1: Split into 60% Train and 40% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

# Step 2: Split the 40% Temp into two equal halves (20% Meta, 20% Test)
X_meta, X_test, y_meta, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Recombine X and y to save clean CSV files
train_df = pd.concat([X_train, y_train], axis=1)
meta_df = pd.concat([X_meta, y_meta], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

# Save to your data/processed folder
train_df.to_csv("D:\HOPE AI\ML model Failure Prediction System\Datasets\Preprocessed/train_60.csv", index=False)
meta_df.to_csv("D:\HOPE AI\ML model Failure Prediction System\Datasets\Preprocessed/meta_20.csv", index=False)
test_df.to_csv("D:\HOPE AI\ML model Failure Prediction System\Datasets\Preprocessed/test_20.csv", index=False)

print(f"Train (60%): {train_df.shape}")
print(f"Meta  (20%): {meta_df.shape}")
print(f"Test  (20%): {test_df.shape}")

Train (60%): (6000, 11)
Meta  (20%): (2000, 11)
Test  (20%): (2000, 11)
